1. S - Sample (Amostragem)O objetivo primário é garantir que o algoritmo aprenda padrões reais de mercado. Precisamos isolar uma base Out-of-Time (OOT) para simular o mundo real e estratificar o treino para manter a proporção exata da variável resposta.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Carregamento da Base de Dados
df = pd.read_csv('abt_churn.csv')

# Conversão da coluna de data
df['dtRef'] = pd.to_datetime(df['dtRef'])

# 1. Out-of-Time (OOT): Isolamento absoluto de dados cronologicamente recentes
max_date = df['dtRef'].max()
oot_cutoff = max_date - pd.DateOffset(months=1)

df_oot = df[df['dtRef'] >= oot_cutoff].copy()
df_model = df[df['dtRef'] < oot_cutoff].copy()

# 2. Divisão de Variáveis (Features vs. Target)
target = 'flagChurn'
ignored_cols = ['flagChurn', 'dtRef', 'idUsuario']

X = df_model.drop(columns=ignored_cols)
y = df_model[target]

X_oot = df_oot.drop(columns=ignored_cols)
y_oot = df_oot[target]

# 3. Train/Test Split com Estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Variáveis criadas com sucesso!")
print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas | OOT: {X_oot.shape[0]} linhas")

Variáveis criadas com sucesso!
Treino: 3793 linhas | Teste: 949 linhas | OOT: 754 linhas


2. E - Explore (Exploração)
Esta etapa é o "raio-X" da base. É crucial que a investigação estatística seja executada exclusivamente na base de Treino, evitando o Data Leakage.

In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

# 1. Auditoria de Missings [cite: 80]
missings = X_train.isnull().sum()
print("Variáveis com valores nulos:\n", missings[missings > 0])
# Decisão Metodológica: O tratamento será encapsulado no Pipeline (Step M) usando SimpleImputer,
# para garantir que os valores calculados (como a média) venham apenas do Treino.

# 2. Análise Bivariada [cite: 83]
# Juntamos o target temporariamente apenas para análise
df_explore = X_train.copy()
df_explore['target_churn'] = y_train

# Comparando o comportamento das variáveis explicativas [cite: 270]
bivariada = df_explore.groupby('target_churn').mean().T
print("\nAnálise Bivariada (Médias por Classe):\n", bivariada.head())

# 3. Feature Importance Preliminar com Árvore de Decisão [cite: 82, 271]
# Preenchendo nulos temporariamente com -999 apenas para rodar a árvore de exploração
arvore_exploratoria = DecisionTreeClassifier(max_depth=4, random_state=42)
arvore_exploratoria.fit(X_train.fillna(-999), y_train)

importancias = pd.Series(
    arvore_exploratoria.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("\nTop 5 Variáveis Mais Importantes[cite: 272]:\n", importancias.head())

Variáveis com valores nulos:
 Series([], dtype: int64)

Análise Bivariada (Médias por Classe):
 target_churn                   0           1
qtdeTransacoes        299.147932   52.448216
qtdeDias               28.256448    9.346375
mediaTransacoesDias     6.277947    4.038429
saldoPontos          1382.591727  444.886076
qtdePontosPos        2864.883212  674.762946

Top 5 Variáveis Mais Importantes[cite: 272]:
 qtdeDiasD14                0.668513
qtdeDiasUltimaTransacao    0.138130
propAvgQtdeDias            0.132835
qtdeDiasD28                0.038038
qtdePontosPos              0.013145
dtype: float64


3. M - Modify (Modificação)
Aqui, fazemos a transformação sintética e matemática da realidade. Ao invés de executar transformações soltas, utilizaremos a classe Pipeline para blindar o modelo contra quebras durante o deploy.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer
from sklearn.impute import SimpleImputer

# Identificação dinâmica de tipos de colunas
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Decisão Metodológica: Discretização de uma variável contínua importante [cite: 276]
# Supondo que 'recency' (dias desde a última venda) seja altamente preditiva.
features_to_bin = ['recency'] if 'recency' in num_features else []
num_features_standard = [f for f in num_features if f not in features_to_bin]

# Construção dos Transformadores
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

bin_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('binner', KBinsDiscretizer(n_bins=5, encode='onehot', strategy='quantile'))
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconhecido')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Engenharia do Motor de Modificações [cite: 129]
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features_standard),
    ('bin', bin_transformer, features_to_bin),
    ('cat', cat_transformer, cat_features)
])

4. M - Model (Modelagem)
O epicentro da inteligência artificial. Vamos orquestrar uma batalha de parâmetros utilizando Cross-Validation para encontrar o "Modelo Campeão".

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Encapsulamento de Governança: Modify + Model juntos [cite: 171]
pipeline_completo = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classificador', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Grade Tática de Batalha (Hyperparameter Tuning) [cite: 175]
parametros = {
    'classificador__n_estimators': [100, 200],
    'classificador__max_depth': [5, 10, None],
    'classificador__min_samples_leaf': [2, 5]
}

# Execução exaustiva via Validação Cruzada (CV=5) [cite: 179]
grid = GridSearchCV(
    pipeline_completo,
    param_grid=parametros,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

# O treinamento rigoroso ocorre aqui
print("Iniciando treinamento via GridSearch...")
grid.fit(X_train, y_train)

print(f"Melhores parâmetros encontrados: {grid.best_params_}")

Iniciando treinamento via GridSearch...
Melhores parâmetros encontrados: {'classificador__max_depth': 5, 'classificador__min_samples_leaf': 2, 'classificador__n_estimators': 200}


5. A - Assess (Avaliação)
A auditoria financeira e estatística implacável. Avaliamos o modelo nas três bases para confirmar a estabilidade contra Overfitting e traduzimos a matemática para impacto de negócio (Gains/Lift).

In [9]:
from sklearn.metrics import roc_auc_score, accuracy_score
import pickle

# 1. Métricas de Performance [cite: 293]
def avaliar_modelo(nome, X, y_true):
    y_pred = grid.predict(X)
    y_prob = grid.predict_proba(X)[:, 1]
    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    print(f"[{nome}] ROC AUC: {auc:.4f} | Acurácia: {acc:.4f}")
    return y_prob

print("\n--- Auditoria de Performance ---")
prob_train = avaliar_modelo("Treino", X_train, y_train)
prob_test  = avaliar_modelo("Teste ", X_test, y_test)
prob_oot   = avaliar_modelo("OOT   ", X_oot, y_oot)

# 2. Análise de Negócio (Decis / Gains) [cite: 296, 297]
# Ordenando a base de teste pela probabilidade de churn
df_gains = pd.DataFrame({'true_churn': y_test, 'probabilidade': prob_test})
df_gains = df_gains.sort_values(by='probabilidade', ascending=False)
df_gains['decil'] = pd.qcut(range(len(df_gains)), 10, labels=False) + 1

# Agrupando por decil para ver a captura antecipada de eventos [cite: 208]
gains_table = df_gains.groupby('decil')['true_churn'].sum().reset_index()
gains_table['taxa_captura_acumulada'] = gains_table['true_churn'].cumsum() / df_gains['true_churn'].sum() * 100

print("\n--- Lift / Gains de Negócio (Base de Teste) ---")
print("Top 10% (Decil 1) de Risco captura: {:.2f}% dos Churners".format(gains_table.iloc[0]['taxa_captura_acumulada']))
print("Top 20% (Decil 2) de Risco captura: {:.2f}% dos Churners".format(gains_table.iloc[1]['taxa_captura_acumulada']))
print("Top 30% (Decil 3) de Risco captura: {:.2f}% dos Churners".format(gains_table.iloc[2]['taxa_captura_acumulada']))

# 3. Serialização: O Artefato Binário [cite: 188, 210]
# Exportando o motor antifraude/churn pronto para a infraestrutura de TI [cite: 242]
with open('motor_churn_semma_v1.pkl', 'wb') as file:
    pickle.dump(grid.best_estimator_, file)

print("\nArtefato serializado com sucesso: motor_churn_semma_v1.pkl")


--- Auditoria de Performance ---
[Treino] ROC AUC: 0.8447 | Acurácia: 0.7590
[Teste ] ROC AUC: 0.8109 | Acurácia: 0.7313
[OOT   ] ROC AUC: 0.8432 | Acurácia: 0.7706

--- Lift / Gains de Negócio (Base de Teste) ---
Top 10% (Decil 1) de Risco captura: 17.01% dos Churners
Top 20% (Decil 2) de Risco captura: 33.56% dos Churners
Top 30% (Decil 3) de Risco captura: 48.74% dos Churners

Artefato serializado com sucesso: motor_churn_semma_v1.pkl
